In [54]:
# !pip install tensorly
# !pip install tensorly-torch

In [55]:
import torch
import torch.nn as nn
class reshape(nn.Module):
    '''
    reshapes the 3-order tensor into 6-order tensor

    ----------
    split : list
        split indices to be applied to each mode of the 3-order tensor

    map_type : int
        based on attached - 1 or compressed - 2 splitting method

    device : str
        operation device, default value is cpu


    inputs a 3-order torch.tensor

    returns a 6-order torch.tensor
    '''
    def __init__(self, split, map_type=1, device='cpu'):
        super(reshape, self).__init__()

        self.split = split
        self.map_type = map_type
        self.device = device

    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []

        if self.map_type == 1:
            # Approach 1 : Attached
            C_indices, H_indices, W_indices = [
                [sum(dim // self.split[i] for _ in range(j)) for j in range(self.split[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]

            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1],
                                           H_indices[j]:H_indices[j+1],
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            # Approach 2 : Compressed
            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            C_stride_indices = torch.arange(i, C, self.split[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.split[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.split[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)

        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.split[0] * self.split[1] * self.split[2])
        result = torch.cat(chunks).view(
            batch_size, self.split[0], self.split[1], self.split[2],
            *chunks[0].shape[1:])
        return result

    def forward(self, x):
        chunks = self.split_into_chunks(x)
        output = self.stack_chunks_to_form_tensor(chunks)
        return output

In [56]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

In [57]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

device  = 'cpu'
print(device)

cpu


In [58]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 8

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [59]:
def topk_accuracy(outputs, targets, topk=(1,)):
    '''
    calculates top-k accuracy

    ----------
    outputs : torch.tensor

    targets : torch.tensor

    topk  : tuple
        calculates top k accuracy given outpurs and targets


    reutrns a python dictionary of top i <= k accuracies

    '''
    maxk = max(topk)
    _, topk_indices = torch.topk(input=outputs, k=maxk, dim=1, largest=True, sorted=True)
    correct = topk_indices.eq(targets.view(-1, 1).expand_as(topk_indices))
    accuracies = {}
    for k in topk:
        correct_k = correct[:,:k].float().sum()
        accuracies[k] = {'correct':correct_k, 'accuracy': (correct_k / outputs.shape[0]) * 100.0}
    return accuracies

In [60]:
def cp(module):
  return sum(p.numel() for p in module.parameters())

In [61]:
def print_gpu_memory_usage(stage):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    string = f'{stage} - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB'
    print(string)
    return string


In [62]:
def append_to_file(file_name, text):
    with open(file_name, 'a') as file:
        file.write(text + '\n')

# FC layers

In [63]:
class CNN1(nn.Module):
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8,256, bias = False)
        self.fc2 = nn.Linear(256,10, bias = False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model1 = CNN1().to(device)


In [64]:
classifier1 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256, bias = False),
    nn.Linear(256, 10, bias = False)
)

print(cp(classifier1))

append_to_file(file_name='TCL_report.txt', text=f'FC classifier # parameters {cp(classifier1)}')

1051136


In [65]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters())

In [66]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model1.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model1(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
    
        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train,forward_time,backward_time

def test_epoch(loader, epoch):
    model1.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [67]:
n_epoch = 10
flag = 1
forward = []
backward = []
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train, forward_time , backward_time = train_epoch(train_loader, epoch)
    forward.append(forward_time)
    backward.append(backward_time)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')

append_to_file(file_name='TCL_report.txt', text=f'FC took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'FC had {string}')
append_to_file(file_name='TCL_report.txt', text=f'FC last epoch result:\n{report_train}\n{report_test}')
append_to_file(file_name='TCL_report.txt', text=f'\n{forward}\n{backward}')

Training for 10 epochs

total forward time : 78.27643013000488, total backward time : 47.714494466781616
Train epoch 1: top1=0.5516999959945679%, top2=0.7484999895095825%, top3=0.845300018787384%, top4=0.9032599925994873%, top5=0.9411399960517883%, loss=0.1551713084191084, time=135.33548998832703s
Test epoch 1: top1=0.6391000151634216%, top2=0.804099977016449%, top3=0.8884000182151794%, top4=0.9315000176429749%, top5=0.9603000283241272%, loss=0.1294548107072711, time=5.231212377548218s
Memory Usage  - Allocated: 16.45 MB, Reserved: 70.00 MB
total forward time : 80.90447211265564, total backward time : 47.810795545578
Train epoch 2: top1=0.6875399947166443%, top2=0.8512399792671204%, top3=0.9183599948883057%, top4=0.9553400278091431%, top5=0.9752399921417236%, loss=0.11048819342851639, time=138.00105595588684s
Test epoch 2: top1=0.6820999979972839%, top2=0.8481000065803528%, top3=0.9175000190734863%, top4=0.9537000060081482%, top5=0.9740999937057495%, loss=0.11428300122879445, time=4.48

In [68]:
import numpy as np
print(np.array(forward).mean() ,  np.array(forward).var())
print(np.array(backward).mean() ,  np.array(backward).var())

80.74990792274475 7.975742142989894
48.79167239665985 8.253856720052578


# TCL from Tensorly

In [69]:
class CNN2(nn.Module):
    def __init__(self):
        super(CNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.fc2(x)
        return x


model2 = CNN2().to(device)


In [70]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

In [71]:
classifier2 = nn.Sequential(
    TCL(input_shape = (64,8,8), rank = (64,2,2)),
    nn.Linear(256,10)
)

print(cp(classifier2))
append_to_file(file_name='TCL_report.txt', text=f'TCL Tensorly classifier # parameters {cp(classifier2)}')

6698


In [72]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters())

In [73]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model2.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model2(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
        

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train, forward_time, backward_time

def test_epoch(loader, epoch):
    model2.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [74]:
n_epoch = 10
flag = 1
forward = []
backward = []
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train, forward_time, backward_time = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    forward.append(forward_time)
    backward.append(backward_time)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly last epoch result:\n{report_train}\n{report_test}')
append_to_file(file_name='TCL_report.txt', text=f'\n{forward}\n{backward}')

Training for 10 epochs

total forward time : 21.155104398727417, total backward time : 26.58524203300476
Train epoch 1: top1=0.46786001324653625%, top2=0.6704999804496765%, top3=0.7862200140953064%, top4=0.860040009021759%, top5=0.909820020198822%, loss=0.18480843392193316, time=52.31910800933838s
Test epoch 1: top1=0.5654000043869019%, top2=0.7638000249862671%, top3=0.8600000143051147%, top4=0.9157000184059143%, top5=0.9513999819755554%, loss=0.1512114302277565, time=3.9546406269073486s
Memory Usage  - Allocated: 16.45 MB, Reserved: 70.00 MB
total forward time : 15.415389060974121, total backward time : 20.724430084228516
Train epoch 2: top1=0.6062999963760376%, top2=0.7935000061988831%, top3=0.8802199959754944%, top4=0.9303600192070007%, top5=0.9604399800300598%, loss=0.1385765007597208, time=40.37784552574158s
Test epoch 2: top1=0.6376000046730042%, top2=0.815500020980835%, top3=0.8967000246047974%, top4=0.9422000050544739%, top5=0.9682999849319458%, loss=0.12747532711476087, time=5

In [75]:
import numpy as np
print(np.array(forward).mean() ,  np.array(forward).var())
print(np.array(backward).mean() ,  np.array(backward).var())

16.999822354316713 5.398222249049494
21.888568258285524 9.015514726336395


In [76]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

# TCL Method 1 just 3D tensors

In [77]:
class TCL2(nn.Module):
    def __init__(self, input_shape, rank, bias = False):
          super(TCL2, self).__init__()
          #suppose it is 3d :
          self.fc1 = nn.Linear(input_shape[0], rank[0], bias = bias)
          self.fc2 = nn.Linear(input_shape[1], rank[1], bias = bias)
          self.fc3 = nn.Linear(input_shape[2], rank[2], bias = bias)

    def forward(self, x):
          x = self.fc3(x)
          x = rearrange(x, 'b c h w -> b c w h')
          x = self.fc2(x)
          x = rearrange(x, 'b c w h -> b w h c')
          x = self.fc1(x)
          x = rearrange(x, 'b w h c -> b c h w')

          return x

In [78]:
class CNN3(nn.Module):
    def __init__(self):
        super(CNN3, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL2(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.fc2(x)
        return x


model3 = CNN3().to(device)


In [79]:
classifier3 = nn.Sequential(
    TCL2(input_shape = (64,8,8), rank = (64,4,4)),
    nn.Linear(256,10)
)

print(cp(classifier3))
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 classifier # parameters {cp(classifier3)}')

6730


In [80]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters())

In [81]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model3.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1
    forward_time = 0
    backward_time = 0


    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model3(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time += time.time() - s

        s = time.time()
        loss.backward()
        optimizer.step()
        if flag:
          backward_time += time.time() - s
        

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'total forward time : {forward_time}, total backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train, forward_time, backward_time

def test_epoch(loader, epoch):
    model3.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [82]:
n_epoch = 10
flag = 1
forward = []
backward = []
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train, forward_time, backward_time = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    forward.append(forward_time)
    backward.append(backward_time)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 last epoch result:\n{report_train}\n{report_test}')
append_to_file(file_name='TCL_report.txt', text=f'\n{forward}\n{backward}')

Training for 10 epochs

total forward time : 15.346727132797241, total backward time : 19.525683164596558
Train epoch 1: top1=0.48142001032829285%, top2=0.6853399872779846%, top3=0.799019992351532%, top4=0.871940016746521%, top5=0.9188799858093262%, loss=0.1784490416023135, time=39.20867872238159s
Test epoch 1: top1=0.5490999817848206%, top2=0.7501000165939331%, top3=0.8514999747276306%, top4=0.9114000201225281%, top5=0.9466000199317932%, loss=0.15589033411741257, time=4.018383502960205s
Memory Usage  - Allocated: 16.25 MB, Reserved: 70.00 MB
total forward time : 15.97911286354065, total backward time : 22.80257511138916
Train epoch 2: top1=0.6163600087165833%, top2=0.8025400042533875%, top3=0.8848400115966797%, top4=0.9331200122833252%, top5=0.9623600244522095%, loss=0.1346914652338624, time=43.0355544090271s
Test epoch 2: top1=0.6345999836921692%, top2=0.8172000050544739%, top3=0.8953999876976013%, top4=0.9404000043869019%, top5=0.9660000205039978%, loss=0.12832265984639527, time=5.8

In [83]:
import numpy as np
print(np.array(forward).mean() ,  np.array(forward).var())
print(np.array(backward).mean() ,  np.array(backward).var())

16.923086047172546 10.746635371532719
22.351194262504578 14.999720999510638


In [84]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')